# 04 — Early Regime-Aware Trading Signals
## <b> Adrian Vazquez </b>


--- 

Early Features
↓
Predicted Regime
↓
Trading Signal
↓
Accuracy

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from scipy.stats import linregress
early_regime = pd.read_csv("../data/processed/early_regime_dataset.csv")
early_regime.head(2)


,market_id,horizon,question,slug,total_n_obs,early_n_obs,early_obs_share,total_time_seconds,early_time_seconds,early_time_share,...,bayesian_fair_probability,mispricing,abs_mispricing,abs_surprise,cluster,market_regime,early_max_drawdown,early_reversals,early_entropy,target
0,248594,0.25,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,457,0.250274,7163979.0,1825177.0,0.254771,...,0.038462,0.018462,0.018462,0.02,0,Information Processing,0.935484,8,0.661563,1
1,248594,0.50,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,913,0.500000,7163979.0,3553194.0,0.495981,...,0.038462,0.018462,0.018462,0.02,0,Information Processing,0.935484,24,0.693147,1


In [7]:
# Can we generate a valid signal from a highly restricted information set?
df_25 = early_regime[early_regime["horizon"] == 0.25].copy()
# we are going to build the early regime classification system 
# train on the full dataset and make predictions

features = [
    "early_last_probability",
    "early_probability_range",
    "early_realized_volatility",
    "early_trend",
    "early_distance_to_boundary",
    "early_max_drawdown",
    "early_reversals",
    "early_entropy"
]

# libs 
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X = df_25[features]
y = df_25["target"]

loo = LeaveOneOut()

predictions = []

for train_idx, test_idx in loo.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    model = Pipeline([ ("scaler", StandardScaler()), ("logreg", LogisticRegression())])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)[0]
    predictions.append(pred)

df_25["predicted_target"] = predictions

#  Map to regime labels
df_25["predicted_regime"] = np.where(df_25["predicted_target"] == 1,"Information Processing",
                                     "Anchored / Noisy")
# Quick validation
df_25[["market_regime","predicted_regime"]].head(10)

,market_regime,predicted_regime
0,Information Processing,Information Processing
3,Anchored / Noisy,Anchored / Noisy
6,Anchored / Noisy,Anchored / Noisy
9,Information Processing,Information Processing
12,Anchored / Noisy,Anchored / Noisy
15,Anchored / Noisy,Information Processing
18,Anchored / Noisy,Information Processing
21,Anchored / Noisy,Anchored / Noisy
24,Anchored / Noisy,Information Processing
27,Anchored / Noisy,Information Processing


In [8]:
(
    df_25["market_regime"]
    ==
    df_25["predicted_regime"]
).mean()

np.float64(0.8604651162790697)

- So far; we have  Early Features - Predicted Regime - 86% accuracy

# buil the early-regime-aware signal



In [9]:
def early_regime_signal(row):
    if row["predicted_regime"] == "Information Processing":
        return -1 if row["mispricing"] > 0 else 1
    else:
        return 1 if row["mispricing"] > 0 else -1
df_25["signal"] = df_25.apply(early_regime_signal,axis=1)
df_25["prediction"] = (df_25["signal"] == 1).astype(int)

# eval 
from sklearn.metrics import classification_report

print(classification_report(df_25["outcome"],df_25["prediction"],zero_division=0))
(df_25["outcome"]==df_25["prediction"]).mean()

              precision    recall  f1-score   support

           0       0.89      0.83      0.86        29
           1       0.69      0.79      0.73        14

    accuracy                           0.81        43
   macro avg       0.79      0.81      0.80        43
weighted avg       0.82      0.81      0.82        43



np.float64(0.813953488372093)

we have : 
| Strategy                 | Accuracy  |
| ------------------------ | --------- |
| Original Bayesian        | **20.9%** |
| Contrarian Bayesian      | **79.1%** |
| Regime-Aware (Oracle)    | **86.0%** |  # <- NoteBook 3
| Early-Regime-Aware (25%) | **81.4%** |

We have oracle regime (all observed) : 86% accuracy 
then, we replace real regime by predict observed path regime  using only 25% of life from market 

And  get : 81.4% 

Early market dynamics contain sufficient information to identify market regimes before resolution, allowing a regime-aware Bayesian mispricing signal to retain approximately 93% of the performance of an oracle regime classifier.

- We found a very promising quantitative hypothesis.

,market_id,horizon,question,slug,total_n_obs,early_n_obs,early_obs_share,total_time_seconds,early_time_seconds,early_time_share,...,bayesian_fair_probability,mispricing,abs_mispricing,abs_surprise,cluster,market_regime,early_max_drawdown,early_reversals,early_entropy,target
0,248594,0.25,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,457,0.250274,7163979.0,1825177.0,0.254771,...,0.038462,0.018462,0.018462,0.0200,0,Information Processing,0.935484,8,0.661563,1
1,248594,0.50,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,913,0.500000,7163979.0,3553194.0,0.495981,...,0.038462,0.018462,0.018462,0.0200,0,Information Processing,0.935484,24,0.693147,1
2,248594,0.75,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,1370,0.750274,7163979.0,5392799.0,0.752766,...,0.038462,0.018462,0.018462,0.0200,0,Information Processing,0.946237,30,0.689944,1
3,249778,0.25,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,23,6,0.260870,100797.0,18011.0,0.178686,...,0.600000,0.050000,0.050000,0.4500,1,Anchored / Noisy,0.000000,0,0.000000,0
4,249778,0.50,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,23,12,0.521739,100797.0,43197.0,0.428554,...,0.600000,0.050000,0.050000,0.4500,1,Anchored / Noisy,0.000000,0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,255337,0.50,Fed rate cut by December 18?,fed-rate-cut-by-december-18,4655,2328,0.500107,16761599.0,8380800.0,0.500000,...,0.900000,-0.099500,0.099500,0.0005,0,Information Processing,0.556842,123,0.689701,1
125,255337,0.75,Fed rate cut by December 18?,fed-rate-cut-by-december-18,4655,3492,0.750161,16761599.0,12574799.0,0.750215,...,0.900000,-0.099500,0.099500,0.0005,0,Information Processing,0.556842,248,0.692934,1
126,255410,0.25,US inflation >0.4% from Feb to March 2024?,us-inflation-0pt4-from-feb-to-march-2024,8,2,0.250000,25199.0,3599.0,0.142823,...,0.600000,0.065000,0.065000,0.5350,1,Anchored / Noisy,0.000000,0,-0.000000,0
127,255410,0.50,US inflation >0.4% from Feb to March 2024?,us-inflation-0pt4-from-feb-to-march-2024,8,4,0.500000,25199.0,10799.0,0.428549,...,0.600000,0.065000,0.065000,0.5350,1,Anchored / Noisy,0.026316,1,0.636514,0
